# AdaBoost Classifier — ML Algorithms Lab

## Hyperparameters taken from the supplied cheat sheet

The cheat sheet's **AdaBoost parameter grid** specifies:

- **`n_estimators`: `[50, 100, 200]`**
- **`learning_rate`: `[0.01, 0.1, 1]`**
- **`estimator`:**
  - `DecisionTreeClassifier(max_depth=1, random_state=42)`
  - `DecisionTreeClassifier(max_depth=2, random_state=42)`

These give **18 combinations**. The notebook uses the current scikit-learn parameter name `estimator` (rather than the older `base_estimator` name). If the installed scikit-learn version is older, a compatibility fallback is included.

The notebook includes leakage-safe preprocessing, GridSearchCV, cross-validation, broad classification metrics, visualizations, feature importance where available, learning curves, and paired significance testing against a default AdaBoost baseline.

**Important:** the numeric grid and the two decision-tree estimator configurations above are from the cheat sheet. `random_state=42` is also specified inside each decision tree in the cheat sheet. Other random-state settings in the notebook are used for reproducible data splits and CV. The notebook is syntax-checked but has not been run on your dataset.


In [ ]:
# ============================================================
# 1. Imports and compatibility
# ============================================================
import os, glob, time, warnings, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, label_binarize
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, classification_report, ConfusionMatrixDisplay,
    log_loss, roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.base import clone

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Imports loaded. random_state =", RANDOM_STATE)

# scikit-learn compatibility: newer versions use estimator=; older versions use base_estimator=.
ADA_USES_ESTIMATOR = "estimator" in inspect.signature(AdaBoostClassifier).parameters
ADA_ESTIMATOR_KEY = "estimator" if ADA_USES_ESTIMATOR else "base_estimator"
print("AdaBoost base-tree parameter name:", ADA_ESTIMATOR_KEY)

# ============================================================
# 2. Load CSV
# ============================================================
csv_files = glob.glob("*.csv")
if len(csv_files) == 1:
    default_path = csv_files[0]
    print("Detected CSV:", default_path)
    path = input("Press Enter to use it, or enter another CSV path: ").strip() or default_path
elif len(csv_files) > 1:
    print("CSV files in current directory:", csv_files)
    path = input("Enter CSV path: ").strip()
else:
    path = input("Enter CSV path: ").strip()

if not os.path.exists(path):
    raise FileNotFoundError(f"Dataset not found: {path}")
df = pd.read_csv(path)
print("Dataset shape:", df.shape)
display(df.head())
print("\nData types:")
df.info()
print("\nDescriptive statistics:")
display(df.describe(include="all").T)
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(30))
print("Duplicate rows:", df.duplicated().sum())

# ============================================================
# 3. Select target, clean, and explore
# ============================================================
target = input("Enter the exact target/class column name: ").strip()
if target not in df.columns:
    raise ValueError(f"Target '{target}' not found. Available columns: {list(df.columns)}")
df = df.drop_duplicates().dropna(subset=[target]).copy()
y = df[target].copy()
X = df.drop(columns=[target]).copy()

# Heuristic identifier removal; inspect this output and restore columns if they are meaningful.
id_like = [c for c in X.columns if (
    c.lower() in {"id", "index", "row_id", "customerid", "customer_id", "record_id"} or
    (X[c].nunique(dropna=True) >= 0.98 * max(len(X), 1) and
     (pd.api.types.is_integer_dtype(X[c]) or X[c].dtype == "object"))
)]
if id_like:
    print("Potential identifier columns excluded (review):", id_like)
    X = X.drop(columns=id_like)
if y.nunique() < 2:
    raise ValueError("Classification requires at least two target classes.")
print("Feature matrix:", X.shape, "| Classes:", y.nunique())
display(y.value_counts().rename("class_count").to_frame())

plt.figure(figsize=(8,4))
y.astype(str).value_counts().plot(kind="bar")
plt.title("Target class distribution"); plt.xlabel("Class"); plt.ylabel("Count")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
if num_cols:
    X[num_cols].hist(figsize=(12, max(4, 2.5*((len(num_cols)+3)//4))), bins=20)
    plt.suptitle("Numeric feature distributions"); plt.tight_layout(); plt.show()
    if len(num_cols) >= 2:
        corr = X[num_cols].corr(numeric_only=True)
        plt.figure(figsize=(9,6)); plt.imshow(corr, aspect="auto"); plt.colorbar(label="Correlation")
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title("Numeric feature correlation matrix"); plt.tight_layout(); plt.show()
for col in cat_cols[:6]:
    plt.figure(figsize=(8,3))
    X[col].astype(str).value_counts().head(15).plot(kind="bar")
    plt.title(f"Most frequent categories: {col}")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

# ============================================================
# 4. Split and preprocessing
# ============================================================
if int(y.value_counts().min()) < 2:
    raise ValueError("At least one class has fewer than 2 examples; stratified CV is not reliable.")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
min_train_class = int(y_train.value_counts().min())
n_splits = min(5, min_train_class)
if n_splits < 2:
    raise ValueError("Insufficient examples per class for cross-validation.")
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
print("Training rows:", len(X_train), "| Test rows:", len(X_test), "| CV folds:", n_splits)

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])
transformers = []
if num_cols:
    transformers.append(("num", numeric_pipe, num_cols))
if cat_cols:
    transformers.append(("cat", categorical_pipe, cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop", sparse_threshold=0)

# ============================================================
# 5. AdaBoost and exact cheat-sheet grid
# ============================================================
# Cheat sheet values:
# n_estimators: [50, 100, 200]
# learning_rate: [0.01, 0.1, 1]
# estimator: DecisionTreeClassifier(max_depth=1, random_state=42)
#            DecisionTreeClassifier(max_depth=2, random_state=42)
tree_estimators = [
    DecisionTreeClassifier(max_depth=1, random_state=42),
    DecisionTreeClassifier(max_depth=2, random_state=42)
]
param_grid = {
    f"model__{ADA_ESTIMATOR_KEY}": tree_estimators,
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 1]
}
print("Cheat-sheet grid combinations: 2 estimators × 3 n_estimators × 3 learning rates = 18")

# Some older scikit-learn versions use algorithm='SAMME.R' by default; current versions
# may not expose `algorithm`. Leave it at the library default for version compatibility.
ada = AdaBoostClassifier(random_state=RANDOM_STATE)
pipeline = Pipeline([("prep", preprocessor), ("model", ada)])

# ============================================================
# 6. GridSearchCV
# ============================================================
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted"
}
search = GridSearchCV(
    estimator=pipeline, param_grid=param_grid, scoring=scoring,
    refit="f1_macro", cv=cv, n_jobs=-1, error_score=np.nan,
    return_train_score=True
)
t0 = time.time()
search.fit(X_train, y_train)
elapsed = time.time() - t0
best_model = search.best_estimator_
print(f"Grid search elapsed time: {elapsed:.1f} seconds")
print("Best parameters:", search.best_params_)
print("Best mean CV macro-F1:", round(search.best_score_, 5))

cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_f1_macro")
display(cv_results[[
    "rank_test_f1_macro", "mean_test_f1_macro", "std_test_f1_macro",
    "mean_test_accuracy", "mean_test_balanced_accuracy",
    "mean_test_precision_macro", "mean_test_recall_macro",
    "mean_test_f1_weighted", "params"
]].head(18))

plt.figure(figsize=(9,4))
top = cv_results.head(18)
plt.plot(range(1, len(top)+1), top["mean_test_f1_macro"], marker="o")
plt.xlabel("Candidate rank"); plt.ylabel("Mean CV macro-F1")
plt.title("AdaBoost grid-search candidates"); plt.tight_layout(); plt.show()

# ============================================================
# 7. Held-out test metrics
# ============================================================
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)
classes = list(best_model.named_steps["model"].classes_)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "precision_weighted": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "matthews_corrcoef": matthews_corrcoef(y_test, y_pred),
    "cohen_kappa": cohen_kappa_score(y_test, y_pred),
    "log_loss": log_loss(y_test, y_prob, labels=classes)
}
try:
    if len(classes) == 2:
        metrics["roc_auc"] = roc_auc_score(y_test, y_prob[:,1])
        metrics["average_precision"] = average_precision_score(y_test, y_prob[:,1])
    else:
        metrics["roc_auc_ovr_macro"] = roc_auc_score(
            y_test, y_prob, labels=classes, multi_class="ovr", average="macro")
        metrics["average_precision_macro"] = average_precision_score(
            label_binarize(y_test, classes=classes), y_prob, average="macro")
except Exception as e:
    print("Some probability-based metrics unavailable:", e)

print("Held-out test metrics:")
display(pd.Series(metrics, name="test_score").to_frame())
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, xticks_rotation=45)
ax.set_title("AdaBoost confusion matrix"); plt.tight_layout(); plt.show()

try:
    if len(classes) == 2:
        RocCurveDisplay.from_predictions(y_test, y_prob[:,1])
        plt.title("AdaBoost ROC curve"); plt.tight_layout(); plt.show()
        PrecisionRecallDisplay.from_predictions(y_test, y_prob[:,1])
        plt.title("AdaBoost precision-recall curve"); plt.tight_layout(); plt.show()
    else:
        from sklearn.metrics import roc_curve
        y_bin = label_binarize(y_test, classes=classes)
        plt.figure(figsize=(8,6))
        for i, cls in enumerate(classes):
            if len(np.unique(y_bin[:,i])) == 2:
                fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
                plt.plot(fpr, tpr, label=f"Class {cls}")
        plt.plot([0,1], [0,1], linestyle="--")
        plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
        plt.title("Multiclass one-vs-rest ROC curves"); plt.legend()
        plt.tight_layout(); plt.show()
except Exception as e:
    print("ROC/PR plots unavailable:", e)

# ============================================================
# 8. Feature importance
# ============================================================
try:
    fitted_prep = best_model.named_steps["prep"]
    fitted_ada = best_model.named_steps["model"]
    feature_names = fitted_prep.get_feature_names_out()
    importances = pd.Series(fitted_ada.feature_importances_, index=feature_names).sort_values(ascending=False)
    display(importances.head(25).rename("importance").to_frame())
    plt.figure(figsize=(9,6))
    importances.head(20).sort_values().plot(kind="barh")
    plt.xlabel("Feature importance"); plt.title("AdaBoost top feature importances")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Feature importance unavailable:", e)

# ============================================================
# 9. Learning curve
# ============================================================
try:
    lc_splits = min(3, int(y_train.value_counts().min()))
    if lc_splits >= 2:
        lc_cv = StratifiedKFold(n_splits=lc_splits, shuffle=True, random_state=RANDOM_STATE)
        sizes, train_scores, val_scores = learning_curve(
            best_model, X_train, y_train, cv=lc_cv, scoring="f1_macro",
            train_sizes=np.linspace(0.3, 1.0, 4), n_jobs=-1, error_score=np.nan
        )
        plt.figure(figsize=(8,5))
        plt.plot(sizes, np.nanmean(train_scores, axis=1), marker="o", label="Train macro-F1")
        plt.plot(sizes, np.nanmean(val_scores, axis=1), marker="o", label="Validation macro-F1")
        plt.xlabel("Training examples"); plt.ylabel("Macro-F1")
        plt.title("AdaBoost learning curve"); plt.legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print("Learning curve skipped:", e)

# ============================================================
# 10. Paired significance test: tuned AdaBoost vs default AdaBoost
# ============================================================
baseline = Pipeline([
    ("prep", preprocessor),
    ("model", AdaBoostClassifier(random_state=RANDOM_STATE))
])
tuned_fold_scores, base_fold_scores = [], []
for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train), start=1):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    tuned_fold = clone(best_model)
    base_fold = clone(baseline)
    tuned_fold.fit(X_tr, y_tr)
    base_fold.fit(X_tr, y_tr)
    tuned_fold_scores.append(f1_score(y_va, tuned_fold.predict(X_va), average="macro", zero_division=0))
    base_fold_scores.append(f1_score(y_va, base_fold.predict(X_va), average="macro", zero_division=0))

print("Tuned fold macro-F1:", np.round(tuned_fold_scores, 4))
print("Default fold macro-F1:", np.round(base_fold_scores, 4))
print("Mean difference (tuned - default):", round(float(np.mean(np.array(tuned_fold_scores)-np.array(base_fold_scores))), 6))
if len(tuned_fold_scores) >= 2:
    print("Paired t-test:", ttest_rel(tuned_fold_scores, base_fold_scores, nan_policy="omit"))
    try:
        print("Wilcoxon signed-rank:", wilcoxon(tuned_fold_scores, base_fold_scores, zero_method="wilcox"))
    except Exception as e:
        print("Wilcoxon unavailable:", e)
print("Caution: CV fold scores are not fully independent; interpret p-values as exploratory.")

# ============================================================
# 11. Export results
# ============================================================
summary = pd.DataFrame([{
    "model": "AdaBoostClassifier",
    "best_params": str(search.best_params_),
    "best_cv_macro_f1": search.best_score_,
    "grid_search_seconds": elapsed,
    **metrics
}])
summary.to_csv("adaboost_classification_summary.csv", index=False)
pd.DataFrame({"actual": y_test.reset_index(drop=True),
              "predicted": pd.Series(y_pred).reset_index(drop=True)}).to_csv(
                  "adaboost_test_predictions.csv", index=False)
cv_results.to_csv("adaboost_grid_search_results.csv", index=False)
print("Saved:")
print("- adaboost_classification_summary.csv")
print("- adaboost_test_predictions.csv")
print("- adaboost_grid_search_results.csv")
print("Notebook complete.")
